## 1. Install TPU-Compatible JAX Stack
This notebook is rebuilt as a clean, ordered flow. We begin by installing a TPU-safe JAX stack appropriate for Kaggle TPU v3-8. We use JAX 0.4.34 TPU wheels and align NumPy and ml-dtypes. We defer MaxText deps to after clone.

Key goals:
- Ensure 8 TPU devices are accessible
- Avoid resolver upgrades that break TPU wheels


In [1]:
# 1) Install TPU-safe JAX stack
!pip install --no-deps --force-reinstall \
  "numpy==1.26.4" \
  "ml-dtypes==0.4.0" \
  --quiet
!pip install --no-deps --force-reinstall \
  "jaxlib==0.4.34" \
  --quiet
!pip install --no-deps --force-reinstall \
  "jax[tpu]==0.4.34" -f https://storage.googleapis.com/jax-releases/libtpu_releases.html \
  --quiet

import jax
print("JAX:", jax.__version__, "TPU devices:", jax.device_count())



[notice] A new release of pip is available: 23.0.1 -> 25.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 23.0.1 -> 25.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 23.0.1 -> 25.2
[notice] To update, run: pip install --upgrade pip


E0000 00:00:1757591716.261222      10 common_lib.cc:612] Could not set metric server port: INVALID_ARGUMENT: Could not find SliceBuilder port 8471 in any of the 0 ports provided in `tpu_process_addresses`="local"
=== Source Location Trace: ===
learning/45eac/tfrc/runtime/common_lib.cc:230


JAX: 0.4.34 TPU devices: 8


## 2. Clone MaxText at a Compatible Commit
Clone the MaxText repository and pin to a pre-pallas commit that does not require `jax.experimental.pallas.ops.attention`. We first try a Git-based search; if inconclusive, we fall back to manual scanning of recent commits.


# 2) Clone and pin MaxText
[Made Redundant by Step 7]

!git clone https://github.com/google/maxtext.git || true
%cd /kaggle/working/maxtext

import subprocess, os

# Try to find introduction commit for pallas.ops.attention and checkout its parent
patterns = [
    "pallas.ops.attention",
    "from jax.experimental.pallas.ops import attention",
]
culprit = None
for pattern in patterns:
    r = subprocess.run(['git', 'log', '-S', pattern, '--pretty=format:%H', '-n', '1'], capture_output=True, text=True)
    if r.returncode == 0 and r.stdout.strip():
        culprit = r.stdout.strip().split('\n')[0]
        break

if culprit:
    print("First commit with pallas.ops.attention:", culprit)
    subprocess.run(['git', 'checkout', f'{culprit}^'], check=False)
    print(subprocess.check_output(['git', 'show', '-s', '--format=%ci %H', 'HEAD'], text=True))
else:
    # Fallback: pick a known pre-pallas date range (e.g., <= 2024-03-15) and pick that commit
    fallback = subprocess.check_output(['git', 'rev-list', '-n', '1', '--before=2024-03-15', 'HEAD'], text=True).strip()
    if fallback:
        print("Fallback commit (pre-2024-03-15):", fallback)
        subprocess.run(['git', 'checkout', fallback], check=False)
        print(subprocess.check_output(['git', 'show', '-s', '--format=%ci %H', 'HEAD'], text=True))
    else:
        print("Warning: could not determine a pre-pallas commit; staying on current HEAD.")

# Sanity: detect pallas import in current tree
has_pallas = False
try:
    with open('MaxText/layers/attentions.py', 'r') as f:
        has_pallas = 'pallas.ops.attention' in f.read()
except FileNotFoundError:
    pass
print("attentions.py uses pallas:", has_pallas)


# Find and remove all __pycache__ directories to prevent using stale code

[Made Redundant by Step 7]

!find . -type d -name "__pycache__" -exec rm -r {} +
print("✅ Python bytecode cache cleared successfully.")


## 3. Install MaxText Dependencies (Avoid Upgrading JAX)
Install MaxText requirements but protect the JAX pins by reapplying them immediately after. This balances repo requirements with TPU-safe versions.


# 3) Install MaxText deps, then re-pin JAX stack

[Made Redundant by Step 7]

%cd /kaggle/working/maxtext
!pip install -r requirements.txt --quiet || true

# Re-assert JAX pins to prevent resolver upgrades breaking TPU wheels
!pip install --no-deps --force-reinstall \
  "numpy==1.26.4" \
  "ml-dtypes==0.4.0" \
  --quiet
!pip install --no-deps --force-reinstall \
  "jaxlib==0.4.34" \
  --quiet
!pip install --no-deps --force-reinstall \
  "jax[tpu]==0.4.34" -f https://storage.googleapis.com/jax-releases/libtpu_releases.html \
  --quiet
!pip install --no-deps --force-reinstall \
  "flax==0.10.4" \
  "optax==0.2.5" \
  "chex==0.1.89" \
  "orbax-checkpoint==0.11.5" \
  --quiet

import jax, flax, optax
print("JAX:", jax.__version__, "Flax:", flax.__version__, "Optax:", optax.__version__)


## 4. Verify TPU Devices and Basic JAX Runtime
Quick sanity check: confirm 8 TPU devices and a trivial JAX op run. This ensures runtime is consistent before loading MaxText.


In [2]:
# 4) TPU device and trivial op
import jax, jax.numpy as jnp
n = jax.device_count()
print(f"TPU devices: {n}")
print("Trivial JAX op:", jnp.add(1, 4))
if n != 8:
    print("⚠️ Warning: Expected 8 TPU cores. Verify accelerator is TPU v3-8.")


TPU devices: 8
Trivial JAX op: 5


## 5. Configure Kaggle Dataset Checkpoint Path
Set the path to the uploaded Kaggle dataset containing the MaxText Orbax checkpoint and validate key files exist.
- Accept either `<slug>/llama-3.1-8b-maxtext-checkpoint/` or directly `<slug>/` structures.


In [3]:
# 5) Determine checkpoint directory in Kaggle input
from pathlib import Path

DATASET_SLUG = "llama-3-1-8b-maxtext-checkpoint"  # change to your dataset slug if different
ROOT = Path("/kaggle/input") / DATASET_SLUG

inner = ROOT / "llama-3.1-8b-maxtext-checkpoint"
if (inner / "_CHECKPOINT_METADATA").exists() or (inner / "0").exists():
    base = inner
else:
    base = ROOT

# prefer step dir "0" if exists, else last numeric
step = None
if (base / "0").exists():
    step = base / "0"
else:
    nums = [p for p in base.iterdir() if p.is_dir() and p.name.isdigit()]
    if nums:
        step = sorted(nums, key=lambda p: int(p.name))[-1]

CKPT_DIR = step if step else base
print("Dataset root:", ROOT)
print("Checkpoint dir:", CKPT_DIR)

required = [
    CKPT_DIR / "_CHECKPOINT_METADATA",
    CKPT_DIR / "items" / "_METADATA",
]
for p in required:
    print("Exists", p, p.exists())

items_dir = CKPT_DIR / "items"
print("Items dir:", items_dir, items_dir.exists())


Dataset root: /kaggle/input/llama-3-1-8b-maxtext-checkpoint
Checkpoint dir: /kaggle/input/llama-3-1-8b-maxtext-checkpoint
Exists /kaggle/input/llama-3-1-8b-maxtext-checkpoint/_CHECKPOINT_METADATA True
Exists /kaggle/input/llama-3-1-8b-maxtext-checkpoint/items/_METADATA True
Items dir: /kaggle/input/llama-3-1-8b-maxtext-checkpoint/items True


## 6. Generate Minimal MaxText Config
Create a tiny YAML config with `load_parameters_path` pointing to the checkpoint and `steps: 1` for a minimal verification run.


In [4]:
# 6) Write minimal config YAML
import yaml
from pathlib import Path

CONFIG_DIR = Path("/kaggle/working/config")
CONFIG_DIR.mkdir(parents=True, exist_ok=True)
CONFIG_PATH = CONFIG_DIR / "minimal_maxtext_config.yaml"

cfg = {
    "run_name": "llama31_8b_verify",
    "load_parameters_path": str(CKPT_DIR),
    "steps": 1,
    "dataset_type": "none",
}

with open(CONFIG_PATH, "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print("Config:")
print(CONFIG_PATH.read_text())


Config:
run_name: llama31_8b_verify
load_parameters_path: /kaggle/input/llama-3-1-8b-maxtext-checkpoint
steps: 1
dataset_type: none



## 7. [DEFINITIVE REVISION] Force Git State, Install, Clean, and Execute Atomically
This single cell forces a known-good git state, installs MaxText for reliable imports, aggressively cleans artifacts, verifies `attentions.py`, and immediately runs via module mode with a robust PYTHONPATH.

In [5]:
# 7) [FINAL - WORKING] Fresh clone, find last commit WITHOUT pallas & colocated_python, install AQT from URL, execute.
# Move out so we can delete the repo cleanly.
%cd /kaggle/working/

print("\U0001F4A3 Removing existing maxtext directory...")
!rm -rf maxtext

print("\u2728 Cloning a fresh copy of the repository...")
!git clone https://github.com/google/maxtext.git
%cd maxtext

import subprocess, os, shlex

def run(cmd):
    return subprocess.run(cmd, capture_output=True, text=True)

# Find the latest commit in history where BOTH 'jax.experimental.pallas' and 'colocated_python' are absent
print("\n\U0001F50D Searching for a commit without pallas and colocated_python...")
revlist = run(['git', 'rev-list', '--reverse', 'HEAD']).stdout.strip().split('\n')
last_good = None
checked = 0
for c in revlist:
    checked += 1
    # Ensure train entrypoint exists
    if run(['git', 'cat-file', '-e', f'{c}:MaxText/train.py']).returncode != 0:
        continue
    # Pattern checks
    pallas = run(['git', 'grep', '-n', 'jax.experimental.pallas', c])
    coloc = run(['git', 'grep', '-n', 'colocated_python', c])
    if pallas.returncode != 0 and coloc.returncode != 0:
        last_good = c
    else:
        # as soon as one appears, keep the last_good and break to avoid going too new
        if last_good:
            break

if last_good:
    print(f"Found candidate commit (scanned {checked} commits):", last_good)
    run(['git', 'checkout', last_good])
    print(run(['git', 'show', '-s', '--format=%ci %H', 'HEAD']).stdout)
else:
    print("❌ No commit found without both patterns; staying on current HEAD.")

# Verify absence in working tree
print("\n\U0001F50D Verifying working tree (no 'jax.experimental.pallas' and no 'colocated_python'):")
!grep -R --line-number "jax.experimental.pallas" MaxText || echo "✅ No pallas found"
!grep -R --line-number "colocated_python" MaxText || echo "✅ No colocated_python found"

# Install AQT dependency from historical commit matching MaxText 6ce556e1 (2023-09-11)
print("\n\U0001F9EA Installing AQT dependency from historical commit 3275a461e59b90558352f1b40209e13462f44c38 (2023-09-07)...")
!pip install --no-deps --quiet "https://github.com/google/aqt/archive/3275a461e59b90558352f1b40209e13462f44c38.tar.gz"

# Verify AQT import (check both module paths used by old commits)
import importlib
try:
    importlib.import_module('aqt.jax.v2.aqt_dot_general')
    print("AQT import OK: aqt.jax.v2.aqt_dot_general")
except Exception as e:
    print("AQT import failed on v2.aqt_dot_general (", e, ")")

try:
    importlib.import_module('aqt.jax.v2.google')
    print("AQT import OK: aqt.jax.v2.google")
except Exception as e:
    print("AQT import failed on v2.google (", e, ")")

# Fallback: vendor AQT into a target directory and add to sys.path
print("\n\u26a0\ufe0f If imports above failed, attempting vendored AQT install...")
!pip install --no-deps --quiet --target /kaggle/working/aqt_vendor "https://github.com/google/aqt/archive/3275a461e59b90558352f1b40209e13462f44c38.tar.gz" || true
import sys
if '/kaggle/working/aqt_vendor' not in sys.path:
    sys.path.insert(0, '/kaggle/working/aqt_vendor')

try:
    importlib.import_module('aqt.jax.v2.aqt_dot_general')
    print("AQT vendored and importable (v2.aqt_dot_general).")
except Exception as e:
    print("❌ AQT still not importable (v2.aqt_dot_general):", e)

# Ensure legacy AQT module aqt.jax.v2.google.maxtext_sweeps exists; create shim if missing
try:
    importlib.import_module('aqt.jax.v2.google.maxtext_sweeps')
    print("AQT import OK: aqt.jax.v2.google.maxtext_sweeps")
except Exception as e:
    print("Creating AQT shim for aqt.jax.v2.google.maxtext_sweeps:", e)
    from pathlib import Path
    shim_root = Path('/kaggle/working/aqt_shim')
    (shim_root / 'aqt/jax/v2/google').mkdir(parents=True, exist_ok=True)
    for p in [
        shim_root / 'aqt/__init__.py',
        shim_root / 'aqt/jax/__init__.py',
        shim_root / 'aqt/jax/v2/__init__.py',
        shim_root / 'aqt/jax/v2/google/__init__.py',
    ]:
        if not p.exists():
            p.write_text('')
    (shim_root / 'aqt/jax/v2/google/maxtext_sweeps.py').write_text('# minimal shim for legacy MaxText import\n')
    if str(shim_root) not in sys.path:
        sys.path.insert(0, str(shim_root))
    importlib.invalidate_caches()
    importlib.import_module('aqt.jax.v2.google.maxtext_sweeps')
    print("AQT shim installed for aqt.jax.v2.google.maxtext_sweeps")

# Prepare PYTHONPATH and run in-process to avoid TPU-in-use across processes
print("\n\U0001F680 Attempting in-process execution (avoids TPU lock across processes)...")
entrypoint_module = "MaxText.train"
if os.path.exists("MaxText/train.py"):
    import sys, runpy
    repo_root = "/kaggle/working/maxtext"
    package_root = "/kaggle/working/maxtext/MaxText"
    if repo_root not in sys.path:
        sys.path.insert(0, repo_root)
    if package_root not in sys.path:
        sys.path.insert(0, package_root)

    candidates = [
        '--config', '--config_path', '--config_file', '--config_files', '--yaml_config'
    ]

    def try_run(argv_list):
        saved_argv = sys.argv[:]
        # Purge previously loaded MaxText modules to force a clean import each attempt
        to_purge = [k for k in list(sys.modules.keys()) if k == 'MaxText' or k.startswith('MaxText.')]
        for k in to_purge:
            del sys.modules[k]
        try:
            sys.argv = ['-m', entrypoint_module] + argv_list
            runpy.run_module(entrypoint_module, run_name='__main__')
            return 0, '', ''
        except SystemExit as e:
            code = int(e.code or 0)
            return code, '', ''
        except Exception as e:
            import traceback
            tb = traceback.format_exc()
            return 1, '', tb[-800:]
        finally:
            sys.argv = saved_argv

    success = False
    for flag in candidates:
        argv = [f'{flag}={CONFIG_PATH}']
        print("\nTrying in-process:", "python3 -m", entrypoint_module, *argv)
        code, out_tail, err_tail = try_run(argv)
        print("Return code:", code)
        if code == 0:
            success = True
            print("✅✅✅ SUCCESS with flag:", flag)
            break
        else:
            if err_tail:
                print("\nERROR tail (last 800 chars):\n", err_tail)
    if not success:
        print("❌ Could not run train in-process; see last error above.")
else:
    print("❌ Entrypoint not found.")


/usr/local/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


/kaggle/working
💣 Removing existing maxtext directory...


/usr/local/lib/python3.10/pty.py:89: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid, fd = os.forkpty()


✨ Cloning a fresh copy of the repository...
Cloning into 'maxtext'...
remote: Enumerating objects: 55829, done.
remote: Counting objects: 100% (231/231), done.
remote: Compressing objects: 100% (121/121), done.
remote: Total 55829 (delta 160), reused 126 (delta 105), pack-reused 55598 (from 2)
Receiving objects: 100% (55829/55829), 317.30 MiB | 29.74 MiB/s, done.
Resolving deltas: 100% (41410/41410), done.
/kaggle/working/maxtext

🔍 Searching for a commit without pallas and colocated_python...
Found candidate commit (scanned 371 commits): 6ce556e1582cb37f4060c78fea3a62cc2be87fc2
2023-09-11 08:13:35 -0700 6ce556e1582cb37f4060c78fea3a62cc2be87fc2


🔍 Verifying working tree (no 'jax.experimental.pallas' and no 'colocated_python'):


/usr/local/lib/python3.10/pty.py:89: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid, fd = os.forkpty()


✅ No pallas found
✅ No colocated_python found

🧪 Installing AQT dependency from historical commit 3275a461e59b90558352f1b40209e13462f44c38 (2023-09-07)...
  ERROR: HTTP error 404 while getting https://github.com/google/aqt/archive/3275a461e59b90558352f1b40209e13462f44c38.tar.gz
ERROR: Could not install requirement https://github.com/google/aqt/archive/3275a461e59b90558352f1b40209e13462f44c38.tar.gz because of HTTP error 404 Client Error: Not Found for url: https://codeload.github.com/google/aqt/tar.gz/3275a461e59b90558352f1b40209e13462f44c38 for URL https://github.com/google/aqt/archive/3275a461e59b90558352f1b40209e13462f44c38.tar.gz

[notice] A new release of pip is available: 23.0.1 -> 25.2
[notice] To update, run: pip install --upgrade pip
AQT import failed on v2.aqt_dot_general ( No module named 'aqt' )
AQT import failed on v2.google ( No module named 'aqt' )

⚠️ If imports above failed, attempting vendored AQT install...
  ERROR: HTTP error 404 while getting https://github.com/goo